# Fitting noisy data revisited

Thinking back to the final part of last week's notebook 10a, where we again fitted lines to noisy data. Here, let's use MCMC to fit the data. We'll do this using a linear fit to demonstrate the method.

## Our data

Here is some noisy data, drawn from an underlying `y = mx + c` distribution, with noise added to it (refer back to the last part of 10a, as we add an unknown error term `f` in addition to some measured error in the data):

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import sys
!{sys.executable} -m pip install corner
import corner

In [ ]:
# Setting a seed - what does this do?
np.random.seed(123)

# The "true" parameters.
m_true = -0.9
b_true = 4.0
f_true = 0.9

# Generate some synthetic data from the model.
N = 35 # number of data points to generate
x = np.sort(10 * np.random.rand(N)) # randomly generate x
yerr = 0.1 + 0.5 * np.random.rand(N) # randomy generate yerr
y = m_true * x + b_true # y = m*x + b
y += np.abs(f_true * y) * np.random.randn(N)
y += yerr * np.random.randn(N)

# plotting this data
plt.errorbar(x, y, yerr=yerr, fmt=".k", capsize=3,label='Our data')

# plotting the model we drew the mock data from
x0 = np.linspace(0, 10, 500)
plt.plot(x0, m_true * x0 + b_true, "r--", alpha=1.0, lw=1, label='True model')

# Always remember to apropriately label your plots!
plt.legend()
plt.xlim(0, 10)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.xlabel("x",fontsize=20)
plt.ylabel("y",fontsize=20)

## The likelihood

To fit this using Bayesian techniques, we will need to define the components of the equation shown when introducting Bayes' Theorem. First up: The likelihood. This often isn't trivial, so let's run through this for this example:

Our model is below. Let's assume this is a good model and use MCMC to find the uncertainty on the parameters.
 $$y = mx+b$$

with variance $s_n^2$

 $$
    s_n^2 = \sigma_n^2+f^2\,(m\,x_n+b)^2 \quad .
$$

This likelihood function is simply a Gaussian where the variance is
underestimated by some fractional amount,  $f$

The linear least squares method used before in the basic MC example assumed the error bars are correct, Gaussian and independent.

We know this is not the case as we set up the model. (Remember we said there was additional intrinsic scatter denoted by $f$).

Unfortunately, there isn't a generalisation of least squares that supports a model like the one that we know to be true.

We can instead write down the log (_natural log_) likelihood function and numerically maximise it. (We use the log simply because products in log are sums and that makes life easier)

In mathematical notation, the correct log likelihood function is:

$$
    \ln\,p(y\,|\,x,\sigma,m,b,f) =
    -\frac{1}{2} \sum_n \left[
        \frac{(y_n-m\,x_n-b)^2}{s_n^2}
        + \ln \left ( 2\pi\,s_n^2 \right )
    \right]
$$

where

$$
    s_n^2 = \sigma_n^2+f^2\,(m\,x_n+b)^2 \quad .
$$

This likelihood function is simply a Gaussian where the variance is
underestimated by some fractional amount:  $f$.


### EXERCISE:

I have written the equation in python below but there are 4 bugs - find and fix them! Please note that we are using `log(f)` as a parameter here (and below) not `f`.

In [ ]:
def log_likelihood(theta, x, y, yerr):
    m, b, log_f = theta # theta is a tuple that contains our parameters to infer
    model = m ** x + b # our simple straight line
    sigma2 = y_err ** 2 + model ** 2 * np.exp(2 * log_f)
    ln_like = -0.5 * np.prod((y - model) ** 2 / sigma2 + np.log10(sigma2))
    return ln_like

## The prior

Next we define the prior term

 - What does this function below do?
    - Discuss it with each other.
 - What are the priors we are setting on our parameters?
    - Are they reasonable?
 - Add comments to the function so that you can remember this later!

In [ ]:
def log_prior(theta):
    m, b, log_f = theta
    if -5.0 < m < 0.5 and 0.0 < b < 10.0 and -10.0 < log_f < 1.0:
        return 0.0
    return -np.inf

## The evidence

We shall ignore the evidence - it is not necessary here - if you don't remember why, go back and read the part about Bayes theorem!

## The posterior

The posterior is obtained from the prior and the likelihood:

In [ ]:
def log_probability(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)

Why have we used logs above? (hint, try doing `np.exp(1000)` and `np.exp(-1000)`) What does this function do?

Now we are ready to write our MH algorithm and fit stuff with our MCMC!

## Recipe for MCMC using metropolis hastings algorithm

Here is a skeleton of the code for doing this. You will need to complete it following the steps below (refer also to the lecture slides).

In [ ]:
# Defines the next sample in the chain
def transition_model(theta):
    return theta_new


def metropolis_hastings(log_probability, transition_model, param_init,iterations,x,y,yerr):
    # log_probability: posterior calculation
    # transition_model(x): gaussian proposal
    # param_init: a starting sample
    # iterations: number of accepted to generated
    # x,y,yerr: the data that we wish to model
    # acceptance_rule(theta_prob,theta_prob_new): decides whether to accept or reject the new sample
    theta = param_init
    accepted = []
    for i in range(iterations):
        # Can you fill me in?
    return np.array(accepted)

### Step 1 - Compute posterior at the initial values

The `metropolis_hastings` function takes the `log_probability` function (which computes the posterior) and `param_init` variables storing initial parameters. Before doing any iteration, compute the (log) posterior probability at these initial values

### Step 2 - Randomly perturb the initial values to a new point

The `transition_model` defines the next step that we might move to. For each of the 3 parameters this function should choose a new value by choosing a point from a Gaussian distribution. The mean of the Gaussian should be the current value of that parameter, and the standard deviation is 0.1. (These values are appropriate for this problem, but in the general case one must be **very careful** in how potential next points are chosen! After coding this up, select the next set of values within the for loop in the `metropolis_hastings` function by calling `transition_model`

### Step 3 - Compute posterior at the new value

We now need (within the for loop) to compute the posterior at the new values. You should already have what you need to do this.

### Step 4 - Compare the two posteriors

To determine the "odds ratio" of the two points, we need to divide the posterior probabilities. But **wait** these are ***log*** posterior probabilities, so we subtract the log of the original values from the log of the new values.

### Step 5 - Do I stay or do I go?

Generate a random number between 1 and 0. If this is bigger than the odds ratio from step 4, then do not move. Otherwise move to the new point. If the odds ratio is already larger than 1 then you will always move. Be careful! You have the log odds ratio from step 4, so you need to do `np.exp(log_odds_ratio)` to get the odds ratio.

### Step 6 - Iterate again

If you accepted the new point add it to the list `accepted` and move back to step 2. Remember to move the new points to be the old points so when comparing posteriors you are always comparing to the previous step, not always to the first.



## Fitting the data

Now that you have your MCMC code, let's run it and determine the parameters of our data

Using some initial starting parameters which are not the correct answer so that we can see it works!


In [ ]:
m_init=m_true - m_true/1.2
b_init=b_true - b_true/1.4
f_init=f_true - f_true/1.1
print(m_true,b_true,f_true, np.log10(f_true))
print(m_init,b_init,f_init, np.log10(f_init))

-0.9 4.0 0.9 -0.045757490560675115
-0.15000000000000002 1.1428571428571428 0.0818181818181819 -1.0871501757188997


In [ ]:
theta_init = [m_init,b_init,f_init]

accepted = metropolis_hastings(
    log_probability,
    transition_model,
    theta_init,
    200000,
    x, y, yerr
)

## Visualization

Below is some code which will visualise the results as the chain moves along for parameter b:
 - Plot it also for parameters a and f
 - What do you notice about the beginning of the chain?
 - Would you include all the samples when quoting your results for the best fit and the 1 $\sigma$ limits on the parameters?
    - Try varying the number of samples plotted in each panel

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(2,1,1)

parameter = 'b'
dimension = 1

n_samples = 10
ax.plot(accepted[0:n_samples,dimension], 'b.', label='Accepted',alpha=0.5)
ax.set_xlabel("Iteration", fontsize=20)
ax.set_ylabel(f"parameter {parameter}", fontsize=20)
ax.set_title(f"MCMC sampling for ${parameter}$ with Metropolis-Hastings. First {n_samples} samples are shown.", fontsize=15)
ax.grid()
ax.legend()

show_final_samples = 150
initial_sample = accepted.shape[0] - show_final_samples

ax2 = fig.add_subplot(2,1,2)
ax2.plot(np.arange(show_final_samples) + initial_sample, accepted[initial_sample:,dimension], 'b.', label='Accepted',alpha=0.5)
ax2.set_xlabel("Iteration", fontsize=20)
ax2.set_ylabel("parameter b", fontsize=20)
ax2.set_title(f"MCMC sampling for $b$ with Metropolis-Hastings, the last {show_final_samples} samples are shown.", fontsize=15)
ax2.grid()
ax2.legend()

fig.tight_layout()

In [ ]:
# This is the number of samples to discard at the start of the MCMC
samples_start = 0

mh_samples = np.array([accepted[samples_start:,0],accepted[samples_start:,1],accepted[samples_start:,2]]).T
labels = ["m", "b", "log(f)"]
fig = corner.corner(mh_samples, labels=labels, truths=[m_true,b_true,np.log(f_true)], quantiles=[0.16, 0.5, 0.84], levels=(1-np.exp(-0.5),), show_titles=True, title_kwargs={"fontsize": 15}, smooth=True)


In [ ]:
# So it's not too much on the plot, we will only plot 100 lines:
inds = np.random.randint(len(mh_samples), size=100)
for ind in inds:
    sample = mh_samples[ind]
    plt.plot(x0, np.dot(np.vander(x0, 2), sample[:2]), "C1", alpha=0.1) # look up the vander function and see what it is doing here
plt.errorbar(x, y, yerr=yerr, fmt=".k", capsize=0, label='data')
plt.plot(x0, m_true * x0 + b_true, "k", label="truth")
plt.plot([],[], 'C1', label='samples')
plt.legend(fontsize=14)
plt.xlim(0, 10)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.xlabel("x", fontsize=20)
plt.ylabel("y", fontsize=20);